# 04 — YOLOv11 Training

Trains a YOLOv11 variant on the unified dataset (notebook 02, health-checked in 03). Default is `yolo11s` — a good balance for an 800-image dataset.

| Variant   | Params | Speed   | When to pick |
|-----------|-------:|---------|--------------|
| `yolo11n` | ~2.6 M | fastest | edge / demo |
| `yolo11s` | ~9.4 M | fast    | **default** |
| `yolo11m` | ~20 M  | moderate| if accuracy plateaus |

In [ ]:
# Install dependencies
# Ultralytics pulls in torch, torchvision, opencv-python, numpy, pandas, matplotlib.
%pip install -q "ultralytics==8.3.*" pandas matplotlib
# On a bare Linux box (no system libGL) OpenCV import can fail — if so, uncomment:
# !apt-get update -qq && apt-get install -y -qq libgl1

In [ ]:
# Imports
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch, shutil

In [ ]:
# Config — paths, training hyperparameters
DATA_YAML   = Path('../data/dataset/data.yaml').resolve()  # built by notebook 02
RUNS_DIR    = Path('../runs/detect')                       # ultralytics writes here
WEIGHTS_DIR = Path('../weights')                           # promoted best.pt lands here

# Training config — change `model` to compare yolo11n / yolo11s / yolo11m
CFG = dict(
    model='yolo11s.pt',
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer='SGD',
    lr0=0.01,
    patience=20,            # early stopping
    project=str(RUNS_DIR),
    name='campus_yolo11s',
    seed=42,
)

assert DATA_YAML.exists(), f'missing {DATA_YAML} — run notebook 02 first'
WEIGHTS_DIR.mkdir(exist_ok=True)
print('CUDA available:', torch.cuda.is_available())
CFG

## 1. Train

In [ ]:
# Train YOLOv11 with the configured hyperparameters
model = YOLO(CFG['model'])
results = model.train(**{k: v for k, v in CFG.items() if k != 'model'})
run_dir = Path(results.save_dir)
print('run dir:', run_dir)

## 2. Training curves
Loss + validation mAP per epoch from `results.csv`.

In [ ]:
# Plot loss + mAP curves from results.csv
df = pd.read_csv(run_dir / 'results.csv')
df.columns = [c.strip() for c in df.columns]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df.plot(x='epoch', y=['train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss'], ax=ax[0])
ax[0].set_title('Loss curves')
df.plot(x='epoch', y=['metrics/mAP50(B)', 'metrics/mAP50-95(B)'], ax=ax[1])
ax[1].set_title('Validation mAP')
plt.tight_layout(); plt.show()

## 3. Promote best checkpoint
Copy `best.pt` from the run directory to `weights/best.pt` for downstream notebooks.

In [ ]:
# Copy best.pt → weights/best.pt
best = run_dir / 'weights' / 'best.pt'
dst = WEIGHTS_DIR / 'best.pt'
shutil.copy2(best, dst)
print('saved', dst)

## 4. Quick sanity inference

In [ ]:
# Predict on a few test images, save annotated outputs under <run_dir>/sanity/
test_imgs = list((Path('../data/dataset/images/test')).glob('*.jpg'))[:4]
m = YOLO(dst)
res = m.predict(source=[str(p) for p in test_imgs], imgsz=CFG['imgsz'], conf=0.25,
                save=True, project=str(run_dir), name='sanity')
print('predictions saved under', run_dir / 'sanity')

### Expected training indicators
- `val/box_loss` decreases smoothly and plateaus
- `mAP@0.5` > 0.80 on val for all 4 classes within ~60–80 epochs
- No runaway gap between train and val loss

Proceed to **notebook 05** for full evaluation.